# Gráficos del informe final — Motos eléctricas (Cuba)

Fuentes: `data/processed/final/motos_final_limpio.json` (513 devenidos tras filtro USD y 100<precio<5000) y `data/processed/final/motos_procesados.json` (300). Solo lectura; guardado en `storytelling/graphs/` (dpi 150).

In [1]:
import json, re, sys
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 150

BASE = None
for nivel in ("data/processed", "../data/processed", "../../data/processed"):
    if Path(nivel).exists():
        BASE = Path(nivel).resolve()
        break
ROOT = BASE.parents[1]
GRAF = ROOT / "storytelling/graphs"
GRAF.mkdir(parents=True, exist_ok=True)
print("GRAF:", GRAF)

GRAF: /home/dagmary/Escritorio/ICD Project/storytelling/graphs


In [2]:
MARCAS = [
    ("topmaq|topmak", "Topmaq"), ("vedca|vedga", "Vedca"), ("unizuki|\\bizuki\\b", "Izuki"),
    ("yoazaky|yoazaki|yoa ?zaky|yoazak", "Yoazaki"), ("mountain", "Mountain"),
    ("bucatti|bucarti|bucati|buccati", "Bucatti"), ("fly ?racing|flywing|\\bfly\\b", "Fly Racing"),
    ("galaxy", "Galaxy"), ("xmox", "Xmox"), ("mishozuki", "Mishozuki"), ("\\bava\\b", "Ava"),
    ("halcon", "Halcon"), ("kvitova|kivitova|kitova", "Kvitova"), ("murasaki", "Murasaki"),
    ("panda", "Panda"), ("\\bmaf\\b", "Maf"), ("\\bmaox\\b", "Maox"), ("\\btorvan\\b", "Torvan"),
    ("mikazuki", "Mikazuki"), ("dcmlz", "DCM-LZ"),
    ("protenzza|protenza|prontenza|prontezza", "Protenzza"), ("voltax", "Voltax"), ("vortax", "Vortax"),
    ("challenger", "Challenger"), ("\\bvolta\\b|volt ?ride", "Volta"), ("xindi", "Xindi"),
    ("leiliang", "Leiliang"), ("karpaty", "Karpaty"), ("meacon", "Meacon"),
    ("zuzuki|suzuki|\\bzuki\\b", "Zuki"), ("\\bsnopy\\b", "Snopy"), ("yadea", "Yadea"),
    ("\\bxtreme\\b", "Xtreme"), ("mavis", "Mavis"), ("\\bgt\\b", "GT"),
]

def _norm(s):
    plano = re.sub(r"[^a-z0-9]+", " ", (s or "").lower())
    return re.sub(r"\s+", " ", plano).strip()

def marca_de_titulo(titulo):
    texto = _norm(titulo)
    for patron, marca in MARCAS:
        if re.search(patron, texto):
            return marca
    return None

In [3]:
# Dataset 513 (precios / marcas / geografía)
with open(BASE / "final/motos_final_limpio.json", encoding="utf-8") as f:
    d513 = pd.DataFrame(json.load(f))
d513 = d513[d513["moneda"] == "USD"]
d513 = d513[(d513["precio"] > 100) & (d513["precio"] < 5000)]
d513["marca"] = d513["titulo"].map(marca_de_titulo)

k = Path("src/scrapers") if Path("src/scrapers/revolico.py").exists() else Path("../src/scrapers")
sys.path.insert(0, str(k.resolve()))
import revolico as rv
listados = sorted((rv.CACHE_DIR / "motos").glob("listado_*.html"))
cat = rv.extraer_catalogos(listados[0].read_text(encoding="utf-8"))
d513["provincia"] = d513["provinceId"].map(cat["provincias"]).fillna("Desconocida")
d513["municipio"] = d513["municipalityId"].map(cat["municipios"]).fillna(d513["provincia"])

rango_etiquetas = ["<500", "500-1500", "1500-2500", "2500-3500", ">3500"]
rangos_limites = [0, 500, 1500, 2500, 3500, np.inf]
d513["rango"] = pd.cut(d513["precio"], bins=rangos_limites, labels=rango_etiquetas)
print("n=513 tras filtro:", len(d513))

n=513 tras filtro: 513


In [4]:
# Dataset 300 (campos técnicos)
with open(BASE / "final/motos_procesados.json", encoding="utf-8") as f:
    d300 = pd.DataFrame(json.load(f))
print("n=300:", len(d300))

n=300: 300


In [5]:
fig, ax = plt.subplots()
ax.hist(d513["precio"], bins=30, color="steelblue", edgecolor="white")
mediana = d513["precio"].median()
ax.axvline(mediana, color="crimson", linestyle="--", linewidth=1.5, label=f"Mediana: {mediana:,.0f} USD")
ax.set_title(f"Distribución de precios — n={len(d513)}")
ax.set_xlabel("Precio (USD)"); ax.set_ylabel("Nº de anuncios")
ax.legend()
fig.tight_layout(); fig.savefig(GRAF / "precio_histograma.png", dpi=150)
plt.close(fig)
print("precio_histograma.png OK")

precio_histograma.png OK


In [6]:
top10 = d513["marca"].value_counts().head(10)
d_top = d513[d513["marca"].isin(top10.index)]
orden = top10.index.tolist()
fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(data=d_top, x="precio", y="marca", order=orden, color="lightsteelblue", ax=ax)
ax.set_title(f"Precio por marca (top 10) — n={len(d_top)}")
ax.set_xlabel("Precio (USD)"); ax.set_ylabel("")
fig.tight_layout(); fig.savefig(GRAF / "precio_por_marca.png", dpi=150)
plt.close(fig)
print("precio_por_marca.png OK")

precio_por_marca.png OK


In [7]:
top_muni = d513["municipio"].value_counts().head(10)
med_muni = d513[d513["municipio"].isin(top_muni.index)].groupby("municipio")["precio"].median().sort_values()
fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(x=med_muni.values, y=med_muni.index, hue=med_muni.index, palette="Blues_d", ax=ax, legend=False)
ax.set_title("Precio mediano por municipio (top 10)")
ax.set_xlabel("Precio mediano (USD)"); ax.set_ylabel("")
for i, v in enumerate(med_muni.values):
    ax.text(v + 10, i, f"{v:,.0f}", va="center", fontsize=8)
fig.tight_layout(); fig.savefig(GRAF / "precio_por_municipio.png", dpi=150)
plt.close(fig)
print("precio_por_municipio.png OK")

precio_por_municipio.png OK


In [8]:
fig, ax = plt.subplots(figsize=(9, 6))
g = sns.barplot(x=top10.index, y=top10.values, hue=top10.index, palette="viridis", ax=ax, legend=False)
ax.set_title("Top 10 marcas por número de anuncios")
ax.set_xlabel("") ; ax.set_ylabel("Nº de anuncios")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")
fig.tight_layout(); fig.savefig(GRAF / "top_marcas.png", dpi=150)
plt.close(fig)
print("top_marcas.png OK")

top_marcas.png OK


/tmp/ipykernel_55059/602099541.py:5: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")


In [9]:
vcs = d513["rango"].value_counts().sort_index()
pct = (vcs / len(d513) * 100)
fig, ax = plt.subplots(figsize=(9, 6))
g = sns.barplot(x=vcs.index.astype(str), y=vcs.values, hue=vcs.index.astype(str), palette="mako", ax=ax, legend=False)
for i, (v, p) in enumerate(zip(vcs.values, pct.values)):
    ax.text(i, v + 8, f"{p:.1f}%", ha="center", fontsize=9)
ax.set_title(f"Rangos de precio — n={len(d513)}")
ax.set_xlabel("Rango (USD)"); ax.set_ylabel("Nº de anuncios")
fig.tight_layout(); fig.savefig(GRAF / "rangos_precio.png", dpi=150)
plt.close(fig)
print("rangos_precio.png OK")

rangos_precio.png OK


In [10]:
m = d300.dropna(subset=["motor_w"]).copy()
m["bucket"] = pd.cut(m["motor_w"], bins=[0, 500, 1000, 1500, 3000, np.inf],
                       labels=["<500", "500-1000", "1000-1500", "1500-3000", ">3000"])
vc = m["bucket"].value_counts().reindex(["<500", "500-1000", "1000-1500", "1500-3000", ">3000"]).fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(9, 6))
g = sns.barplot(x=vc.index.astype(str), y=vc.values, hue=vc.index.astype(str), palette="crest", ax=ax, legend=False)
for i, v in enumerate(vc.values):
    ax.text(i, v + 1, str(v), ha="center", fontsize=9)
ax.set_title(f"Motor (watts) — n={len(m)}")
ax.set_xlabel("Potencia del motor (W)"); ax.set_ylabel("Nº de anuncios")
fig.tight_layout(); fig.savefig(GRAF / "motor_histograma.png", dpi=150)
plt.close(fig)
print("motor_histograma.png OK")

motor_histograma.png OK


In [11]:
a = d300.dropna(subset=["autonomia_km"]).copy()
a["bucket"] = pd.cut(a["autonomia_km"], bins=[0, 40, 80, 120, 160, np.inf],
                       labels=["0-40", "40-80", "80-120", "120-160", ">160"])
vc = a["bucket"].value_counts().reindex(["0-40", "40-80", "80-120", "120-160", ">160"]).fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(9, 6))
g = sns.barplot(x=vc.index.astype(str), y=vc.values, hue=vc.index.astype(str), palette="flare", ax=ax, legend=False)
for i, v in enumerate(vc.values):
    ax.text(i, v + 1, str(v), ha="center", fontsize=9)
ax.set_title(f"Autonomía (km) — n={len(a)}")
ax.set_xlabel("Autonomía (km)"); ax.set_ylabel("Nº de anuncios")
fig.tight_layout(); fig.savefig(GRAF / "autonomia_histograma.png", dpi=150)
plt.close(fig)
print("autonomia_histograma.png OK")

autonomia_histograma.png OK


In [12]:
v = d300.dropna(subset=["voltaje_v"]).copy()
def etq_volt(vol):
    if vol in (48, 60, 72, 96):
        return f"{vol}V"
    return "otros"
v["cat"] = v["voltaje_v"].map(etq_volt)
vc = v["cat"].value_counts()
fig, ax = plt.subplots(figsize=(9, 6))
g = sns.barplot(x=vc.index.astype(str), y=vc.values, hue=vc.index.astype(str), palette="rocket", ax=ax, legend=False)
for i, val in enumerate(vc.values):
    ax.text(i, val + 1, str(val), ha="center", fontsize=9)
ax.set_title(f"Voltaje (V) — n={len(v)}")
ax.set_xlabel("Voltaje") ; ax.set_ylabel("Nº de anuncios")
fig.tight_layout(); fig.savefig(GRAF / "voltaje_histograma.png", dpi=150)
plt.close(fig)
print("voltaje_histograma.png OK")

voltaje_histograma.png OK


In [13]:
tb = d300["tipo_bateria"].fillna("Sin dato")
tb = tb.replace("plomo", "Plomo").replace("litio", "Litio")
vc = tb.value_counts()
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(vc.values, labels=vc.index, autopct="%1.0f%%", startangle=90, colors=["#2ca02c", "#d62728", "#7f7f7f"],
       counterclock=False, textprops={"fontsize": 10})
ax.set_title(f"Tipo de batería — n={len(tb)}")
fig.tight_layout(); fig.savefig(GRAF / "tipo_bateria.png", dpi=150)
plt.close(fig)
print("tipo_bateria.png OK")

tipo_bateria.png OK


In [14]:
tr = d300["transporte_incluido"].fillna(False)
vc = tr.value_counts().reindex([True, False]).fillna(0).astype(int)
pct = (vc / len(tr) * 100)
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(vc.values, labels=["Incluido", "No incluido"], autopct=lambda p: f"{p:.0f}%", startangle=90,
       colors=["#2ca02c", "#d62728"], counterclock=False, textprops={"fontsize": 10})
ax.set_title(f"Transporte incluido — n={len(tr)}")
fig.tight_layout(); fig.savefig(GRAF / "transporte_incluido.png", dpi=150)
plt.close(fig)
print("transporte_incluido.png OK")

transporte_incluido.png OK


In [15]:
sc = d300.dropna(subset=["motor_w", "precio_usd"]).copy()
fig, ax = plt.subplots(figsize=(9, 6))
sns.regplot(data=sc, x="motor_w", y="precio_usd", scatter_kws={"alpha": 0.4, "s": 25},
            line_kws={"color": "crimson"}, ax=ax)
ax.set_title(f"Motor vs precio — n={len(sc)}")
ax.set_xlabel("Motor (W)"); ax.set_ylabel("Precio (USD)")
fig.tight_layout(); fig.savefig(GRAF / "motor_vs_precio.png", dpi=150)
plt.close(fig)
print("motor_vs_precio.png OK")

motor_vs_precio.png OK


In [16]:
sa = d300.dropna(subset=["autonomia_km", "precio_usd"]).copy()
fig, ax = plt.subplots(figsize=(9, 6))
sns.regplot(data=sa, x="autonomia_km", y="precio_usd", scatter_kws={"alpha": 0.4, "s": 25},
            line_kws={"color": "crimson"}, ax=ax)
ax.set_title(f"Autonomía vs precio — n={len(sa)}")
ax.set_xlabel("Autonomía (km)"); ax.set_ylabel("Precio (USD)")
fig.tight_layout(); fig.savefig(GRAF / "autonomia_vs_precio.png", dpi=150)
plt.close(fig)
print("autonomia_vs_precio.png OK")

autonomia_vs_precio.png OK


In [17]:
archivos = sorted(GRAF.glob("*.png"))
print("Gráficos generados:", len(archivos))
for a in archivos:
    print(" ", a.name, f"({a.stat().st_size//1024} KB)")
print("Ubicación:", GRAF)

Gráficos generados: 12
  autonomia_histograma.png (38 KB)
  autonomia_vs_precio.png (50 KB)
  motor_histograma.png (36 KB)
  motor_vs_precio.png (78 KB)
  precio_histograma.png (44 KB)
  precio_por_marca.png (53 KB)
  precio_por_municipio.png (59 KB)
  rangos_precio.png (46 KB)
  tipo_bateria.png (42 KB)
  top_marcas.png (53 KB)
  transporte_incluido.png (42 KB)
  voltaje_histograma.png (36 KB)
Ubicación: /home/dagmary/Escritorio/ICD Project/storytelling/graphs
